# Lab 5: Agent Middleware

**Difficulty: Intermediate | ~40 min | Requires Lab 2**

## Step 1 — Install the required modules

In [1]:
# One command installs all required modules (versions pinned for reproducibility)
!pip install -qU "langchain==1.2.15" "langchain-core==1.2.28" "langchain-openai==1.1.12" "langgraph==1.1.6" "python-dotenv==1.2.2" "pydantic==2.13.4"

## Step 2 — Load your API key

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENROUTER_API_KEY"):
    raise SystemExit("No OPENROUTER_API_KEY found. Add it to .env and restart the kernel.")

## Step 3 — Create the model

In [3]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0,
)

## Step 4 — Build a plain agent (the canvas for middleware)

`create_agent(model)` assembles the bare loop around your model — no tools yet, so
a run is exactly one model call. This is the object middleware will wrap. Because
`create_agent` accepts a model *instance*, every detail from Step 3 (base URL, key,
temperature) carries in automatically.

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    system_prompt="You are a concise assistant. Answer in one short sentence.",
)

Then run it. `invoke` takes the input schema — a list of `("human", text)` tuples —
and returns the full final state. `result["messages"][-1]` is the agent's last message.

In [5]:
result = agent.invoke({"messages": [("human", "Say hello in one short sentence.")]})
print(result["messages"][-1].content)
print(f"Total messages in state: {len(result['messages'])}")

Hello!
Total messages in state: 2


## Step 5 — Prebuilt middleware: PII redaction

`PIIMiddleware("email", strategy="redact", apply_to_input=True)` attaches a
`before_model` hook that scans the newest human message, replaces any detected email
with `[REDACTED_EMAIL]`, and hands the sanitized state to the model. The `block`
strategy *raises* `PIIDetectionError` in `before_model` instead — the run is refused
before a single model call, the right behavior for a hard compliance rule.

In [6]:
from langchain.agents.middleware import PIIMiddleware, PIIDetectionError

pii_redact = create_agent(
    model=model,
    middleware=[PIIMiddleware("email", strategy="redact", apply_to_input=True)],
    system_prompt="You are a helpful assistant.",
)

pii_block = create_agent(
    model=model,
    middleware=[PIIMiddleware("email", strategy="block", apply_to_input=True)],
    system_prompt="You are a helpful assistant.",
)

Now run both. The redact run proves the mechanism: the reply can only reference
`[REDACTED_EMAIL]`, and the "saw" line prints the *actual* message content from the
returned state. The block run is wrapped in `try/except` because refusing the run is
the expected behavior — we catch the error and print a confirmation.

In [7]:
result = pii_redact.invoke(
    {"messages": [("human", "My email is john.smith@example.com. What is my email?")]}
)
print("Model reply:", result["messages"][-1].content)

last_user = [m for m in result["messages"] if m.type == "human"][-1]
print("What the model actually saw:", repr(last_user.content))

try:
    pii_block.invoke({"messages": [("human", "My email is john.smith@example.com")]})
except PIIDetectionError:
    print("block strategy: PIIDetectionError raised before any model call")

Model reply: I don't have access to your email address or any personal information you've shared in previous conversations. For privacy and security reasons, I'm designed to:

1. **Not store or recall personal data** from past interactions unless explicitly provided in the current conversation (and even then, only for the duration of that chat to provide immediate help).
2. **Respect redactions or privacy markers** like `[REDACTED_EMAIL]` — if you've intentionally obscured information, I will treat it as private and not attempt to infer, guess, or reveal it.

If you need to share your email address with me for a specific, legitimate purpose (e.g., to help with account recovery, subscription services, or another task you've initiated), please feel free to type it out directly in your next message. I’ll only use it for the immediate task at hand and won’t retain it afterward.

If this was a test of my privacy safeguards — thank you for being cautious! Protecting personal data is importan

## Step 6 — Prebuilt middleware: model call limit

`ModelCallLimitMiddleware(run_limit=1)` counts model calls per run and jumps the loop
straight to the end with a "limit exceeded" message when the budget is spent. To make
the limit *bite*, the agent needs a reason to call the model twice, so a trivial
`get_weather` tool is added: one model → tool → model round trip.

In [8]:
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"Weather in {city}: 22C, sunny."

budget_agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[ModelCallLimitMiddleware(run_limit=1, exit_behavior="end")],
    system_prompt="Use the weather tool whenever the user asks about weather.",
)

Run it and dump the whole state so the round trip is visible — each line is one
message the loop accumulated.

In [9]:
result = budget_agent.invoke(
    {"messages": [("human", "What is the weather in Paris? Use the get_weather tool.")]}
)
for m in result["messages"]:
    print(f"  {m.type}: {str(m.content)[:70]}")

  human: What is the weather in Paris? Use the get_weather tool.
  ai: 
  tool: Weather in Paris: 22C, sunny.
  ai: Model call limits exceeded: run limit (1/1)


## Step 7 — Custom middleware (class style): logging

Subclass `AgentMiddleware` and override hooks. `before_model` runs before each model
call and prints how many messages the model is about to see; `after_model` runs right
after and prints the reply. Both return `None` — pure observation, the minimal custom
middleware, and the tool you reach for whenever the loop feels like a black box.

In [10]:
from langchain.agents.middleware import AgentMiddleware, AgentState
from langgraph.runtime import Runtime

class LoggingMiddleware(AgentMiddleware):
    def before_model(self, state: AgentState, runtime: Runtime) -> dict | None:
        print(f"  [before_model] calling the model with {len(state['messages'])} message(s)")
        return None

    def after_model(self, state: AgentState, runtime: Runtime) -> dict | None:
        print(f"  [after_model] model replied: {state['messages'][-1].content[:40]!r}")
        return None

Attach and run it — exactly the same `middleware=[...]` slot the prebuilt pieces used.

In [11]:
logging_agent = create_agent(
    model=model,
    middleware=[LoggingMiddleware()],
    system_prompt="Answer in one short sentence.",
)

result = logging_agent.invoke({"messages": [("human", "What is 2 + 2?")]})
print("Final reply:", result["messages"][-1].content)

  [before_model] calling the model with 1 message(s)


  [after_model] model replied: '2\u202f+\u202f2 equals\u202f4.'
Final reply: 2 + 2 equals 4.


## Step 8 — Custom middleware (wrap style): timing

Node hooks see state; wrap hooks *surround the call*. `wrap_model_call(request,
handler)` hands you the model request and a `handler` that actually performs the
call. Measuring the time around `handler(request)` gives a latency meter — something
no prebuilt middleware provides. `time.perf_counter()` is the most precise clock for
short intervals.

In [12]:
import time
from collections.abc import Callable
from langchain.agents.middleware import ModelRequest, ModelResponse

class TimingMiddleware(AgentMiddleware):
    def wrap_model_call(
        self, request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]
    ) -> ModelResponse:
        start = time.perf_counter()
        response = handler(request)
        elapsed = time.perf_counter() - start
        print(f"  [wrap_model_call] model call took {elapsed:.2f} seconds")
        return response

Attach and run — the printed number is the wall-clock time of the model round trip
itself, not the whole agent.

In [13]:
timing_agent = create_agent(
    model=model,
    middleware=[TimingMiddleware()],
    system_prompt="Answer in one short sentence.",
)

result = timing_agent.invoke({"messages": [("human", "What is the capital of France?")]})
print("Final reply:", result["messages"][-1].content)

  [wrap_model_call] model call took 25.00 seconds
Final reply: The capital of France is Paris.


## Step 9 — Custom middleware (decorator style): one-function hooks

For a single small hook, `@before_model` and `@after_model` turn a plain function
into middleware — no class boilerplate. The decorated function keeps the same
signature as the hook methods: `(state, runtime)`, returning a dict or `None`.

In [14]:
from langchain.agents.middleware import before_model, after_model

@before_model
def log_before(state: AgentState, runtime: Runtime) -> dict | None:
    print(f"  [@before_model] {len(state['messages'])} message(s) in state")
    return None

@after_model
def log_after(state: AgentState, runtime: Runtime) -> dict | None:
    print(f"  [@after_model] reply: {state['messages'][-1].content[:30]!r}")
    return None

Attach both and run.

In [15]:
decorated_agent = create_agent(
    model=model,
    middleware=[log_before, log_after],
    system_prompt="Answer in one short sentence.",
)

result = decorated_agent.invoke({"messages": [("human", "Name one color of a banana.")]})
print("Final reply:", result["messages"][-1].content)

  [@before_model] 1 message(s) in state


  [@after_model] reply: 'Yellow.'
Final reply: Yellow.


## Optional Exercise — Stack middleware and watch the hooks compose

Build ONE agent that uses the `get_weather` tool, `LoggingMiddleware()` (Step 7),
`PIIMiddleware("email", strategy="redact", apply_to_input=True)` (Step 5), and
`ModelCallLimitMiddleware(run_limit=1)` (Step 6). Ask it a question that mentions an
email address and the weather, then print what the model actually saw as the final
user message. Verify: (1) the raw email never reaches the model — only
`[REDACTED_EMAIL]`; (2) your `[before_model]` / `[after_model]` log lines appear
around the model calls; (3) if the agent uses the tool, the run ends with the
`Model call limits exceeded: run limit (1/1)` message instead of a final answer.
Everything needed is already defined above — no new imports.

In [16]:
stacked_agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        LoggingMiddleware(),
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        ModelCallLimitMiddleware(run_limit=1, exit_behavior="end"),
    ],
    system_prompt="Use the weather tool whenever the user asks about weather.",
)

result = stacked_agent.invoke(
    {"messages": [("human", "Tell lisa@example.com what the weather in London is. Use the get_weather tool.")]}
)

last_user = [m for m in result["messages"] if m.type == "human"][-1]
print("What the model saw:", repr(last_user.content))
for m in result["messages"]:
    print(f"  {m.type}: {str(m.content)[:70]}")

  [before_model] calling the model with 1 message(s)


  [after_model] model replied: ''
  [before_model] calling the model with 3 message(s)
What the model saw: 'Tell [REDACTED_EMAIL] what the weather in London is. Use the get_weather tool.'
  human: Tell [REDACTED_EMAIL] what the weather in London is. Use the get_weath
  ai: 
  tool: Weather in London: 22C, sunny.
  ai: Model call limits exceeded: run limit (1/1)
